In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import ExoWeave Coupler
from exoweave import ExoCoupler

# Set A&A publication-ready plotting parameters
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.linewidth': 1.2
})


In [2]:
# =============================================================================
# 1. SETUP SUB-NEPTUNE PARAMETERS
# =============================================================================
M_JUP_IN_EARTH = 317.83
target_mass_mjup = 8.0 / M_JUP_IN_EARTH  # Convert 8.0 M_earth to M_jup

base_params = {
    "mass": target_mass_mjup,
    "T_irr": 500.0,              # Surface equilibrium temperature (K)
    "T_int": 100.0,              # Deep internal heat flux (K)
    "Met": 2,                  # Bulk atmospheric metallicity (approx 5x solar)
    "core_mass_earth": 7.9,      # 7.0 M_earth core leaves 1.0 M_earth for the envelope!
    "iron_fraction": 0.33,       # Earth-like rock/iron ratio
    "f_sed": 3.0,
    "kzz": 8.0,
    "debug": False
}

config = {
    "max_iterations": 20,              
    "mass_convergence_threshold": 0.01,
    "p_bottom_bar": 1.0,
    "output_dir": "./outputs/subneptune_test",
    "resolution": 50,           
    "target_resolution": 50    
}

# Define the two distinct architectures
params_sharp = base_params.copy()
params_sharp["sigma_val"] = 0.1  # Sharp Core (Fully convective envelope)

params_fuzzy = base_params.copy()
params_fuzzy["sigma_val"] = 0.30  # Fuzzy Core (Dilute gradient, layered convection)

In [3]:
# =============================================================================
# 2. RUN THE COUPLED MODELS
# =============================================================================
print("🚀 Launching Sharp Core Model (sigma = 0.01)...")
coupler_sharp = ExoCoupler(target_params=params_sharp, config=config)
results_sharp = coupler_sharp.run()

print("\n🚀 Launching Fuzzy Core Model (sigma = 0.30)...")
coupler_fuzzy = ExoCoupler(target_params=params_fuzzy, config=config)
results_fuzzy = coupler_fuzzy.run()

# =============================================================================
# 3. EXTRACT AND PLOT P-T PROFILES
# =============================================================================
if results_sharp['status'] == 'converged' and results_fuzzy['status'] == 'converged':
    print("\n✅ Both models converged! Generating A&A Figure...")
    
    # Extract interior profiles (Modify these keys if your exact DataFrame structure differs)
    # Assuming 'interior_raw' contains 'pressure' (Pa) and 'temperature' (K)
    int_sharp = results_sharp['interior_raw']
    int_fuzzy = results_fuzzy['interior_raw']
    
    # Convert Pa to Bar for plotting
    p_sharp_bar = int_sharp['pressure'].values / 1e5
    t_sharp = int_sharp['temperature'].values
    
    p_fuzzy_bar = int_fuzzy['pressure'].values / 1e5
    t_fuzzy = int_fuzzy['temperature'].values

    # --- PLOTTING ---
    fig, ax1 = plt.subplots(figsize=(8, 6))

    ax1.plot(t_sharp, p_sharp_bar, color='#1f77b4', lw=2.5, label=r'Sharp Core ($\sigma=0.01$)')
    ax1.plot(t_fuzzy, p_fuzzy_bar, color='#d62728', lw=2.5, linestyle='--', label=r'Fuzzy Core ($\sigma=0.30$)')

    ax1.set_yscale('log')
    ax1.invert_yaxis()  # High pressure at the bottom
    
    ax1.set_xlabel('Temperature (K)')
    ax1.set_ylabel('Pressure (bar)')
    ax1.set_title('Internal $P$--$T$ Profile', loc='left', pad=10, fontweight='bold')

    # Formatting the axes for astrophysics standards
    ax1.tick_params(axis='both', which='major', length=6, width=1.2, direction='in', right=True, top=True)
    ax1.tick_params(axis='both', which='minor', length=3, width=1.0, direction='in', right=True, top=True)
    ax1.grid(True, which='major', linestyle=':', alpha=0.6)

    # Annotate the thermal blanketing
    # (You may need to adjust the exact xy coordinates based on where your specific adiabats land!)
    ax1.annotate('Thermal\nBlanketing\n(Entropy Jumps)', 
                 xy=(t_fuzzy[-1] * 0.9, p_fuzzy_bar[-1] * 0.5), 
                 xytext=(t_fuzzy[-1] * 0.5, p_fuzzy_bar[-1] * 0.1),
                 arrowprops=dict(facecolor='black', arrowstyle='->', lw=1.5), va='center')

    ax1.legend(loc='lower left', framealpha=0.9, edgecolor='k')

    plt.tight_layout()
    plt.savefig("hades_coupled_pt_profile.pdf", format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print("\n❌ One or both solvers failed to converge. Cannot plot comparison.")
    print(f"Sharp Status: {results_sharp['status']}")
    print(f"Fuzzy Status: {results_fuzzy['status']}")

INFO: 👢 Bootstrapping initial gravity via 1-bar interior pre-solve...


🚀 Launching Sharp Core Model (sigma = 0.01)...
--- Loading Raw EOS Tables (From Disk) ---
  > Loading Hydrogen...
  > Loading Helium...
  > Loading Water...
  > Loading Rock...
  > Loading Iron...
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


INFO: ✅ Bootstrap successfully locked initial g_1bar = 15.12 m/s²
INFO: 🌌 Grid Setup: Generated mathematical cold-start prior down to 1.0 bars.
INFO: 
INFO: 🔄 ITERATION 1/20 | Target Mass: 0.02517068873297046 M_Jup | g: 15.12 m/s²
INFO: ==================================================
INFO: Starting ExoREM Simulation...
INFO: Generated namelist at /private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpnnsi7kyo/input.nml
INFO: Running Fortran backend from /Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/bin...


RuntimeError: 
========================================
⚠️ EXOREM FAILED TO CONVERGE OR NO OUTPUT GENERATED
========================================
The Fortran code exited normally, but no HDF5 file was created.
This usually means the atmosphere model failed to converge.

--- STDOUT ---
Exo-REM 2.4.2
____

Reading parameters in file '/private/var/folders/ld/pl_0zzs158sb8h6_y3mrd0lm0000gp/T/tmpnnsi7kyo/input.nml'
Loading standard elemental abundances in file '/Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/data/solar_abundances.dat'
Loading thermochemical data...
 Reading thermochemical table in file '/Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/data/thermochemical_tables/gases/Al.tct.dat'
 Reading thermochemical table in file '/Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/data/thermochemical_tables/gases/Ar.tct.dat'
 Reading thermochemical table in file '/Users/cwilkinson/.exowrap/exorem_source/dist/dist/exorem/data/thermochemical_tables/gases/Ca.tct